# 펑션 콜링 적용하기

In [1]:
from datetime import datetime

def get_current_time():
    now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    print(now)
    return now

if __name__ == '__main__':
    get_current_time()

2026-09-20 17:17:13


In [2]:
tools = [
    {
        'type': 'function',
        'function': {
            'name': 'get_current_time',
            'description': '현재 날짜와 시간을 반환합니다.'
        }
    }
]

In [3]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')

client = OpenAI(api_key=api_key)

def get_ai_response(messages, tools=None):
    response = client.chat.completions.create(
        model='gpt-4o',
        messages=messages,
        tools=tools,
    )
    return response

In [4]:
messages = [
    {'role': 'system', 'content': '너는 사용자를 도와주는 상담사야.'}
]

while True:
    user_input = input('사용자\t: ')
    if user_input == 'exit':
        break

    messages.append({'role': 'user', 'content': user_input})

    ai_response = get_ai_response(messages, tools=tools)
    ai_message = ai_response.choices[0].message
    print(ai_message)

    tool_calls = ai_message.tool_calls
    if tool_calls:
        tool_name = tool_calls[0].function.name
        tool_call_id = tool_calls[0].id

        if tool_name == 'get_current_time':
            messages.append({
                'role': 'function',
                'tool_call_id': tool_call_id,
                'name': tool_name,
                'content': get_current_time(),
            })

        ai_response = get_ai_response(messages, tools=tools)
        ai_message = ai_response.choices[0].message

    messages.append(ai_message)

    print('AI\t: ' + ai_message.content)

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_XHBVFYOGj8EURfiGUEzwpmF3', function=Function(arguments='{}', name='get_current_time'), type='function')])
2026-09-20 17:23:29
AI	: 현재 시각은 2026년 9월 20일 오후 5시 23분입니다.


In [5]:
messages = [
    {'role': 'system', 'content': '너는 사용자를 도와주는 상담사야.'}
]

while True:
    user_input = input('사용자\t: ')
    if user_input == 'exit':
        break

    messages.append({'role': 'user', 'content': user_input})

    ai_response = get_ai_response(messages, tools=tools)
    ai_message = ai_response.choices[0].message
    print(ai_message)

    tool_calls = ai_message.tool_calls
    if tool_calls:
        tool_name = tool_calls[0].function.name
        tool_call_id = tool_calls[0].id

        if tool_name == 'get_current_time':
            messages.append({
                'role': 'function',
                'tool_call_id': tool_call_id,
                'name': tool_name,
                'content': get_current_time(),
            })

        ai_response = get_ai_response(messages, tools=tools)
        ai_message = ai_response.choices[0].message

    messages.append(ai_message)

    print('AI\t: ' + ai_message.content)

ChatCompletionMessage(content='안녕하세요! 어떻게 도와드릴까요?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)
AI	: 안녕하세요! 어떻게 도와드릴까요?
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_hWb7Xl0OX22Md9imA5u1fR8o', function=Function(arguments='{}', name='get_current_time'), type='function')])
2026-09-20 17:26:09
AI	: 현재 시간은 한국 기준으로 2026년 9월 20일 오후 5시 26분입니다. 뉴욕은 한국보다 13시간 느리므로, 뉴욕은 현재 오전 4시 26분입니다. 다른 질문이 있으신가요?
ChatCompletionMessage(content='알겠습니다. 나중에 또 필요하시면 언제든지 말씀해 주세요. 좋은 하루 보내세요!', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)
AI	: 알겠습니다. 나중에 또 필요하시면 언제든지 말씀해 주세요. 좋은 하루 보내세요!


# 도시별 시간 알려주기

In [6]:
import pytz

def get_current_time(timezone: str='Asia/Seoul'):
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime('%Y-%m-%d %H:%M:%S')
    now_timezone = f'{now} {timezone}'
    print(now_timezone)
    return now_timezone

In [7]:
tools = [
    {
        'type': 'function',
        'function': {
            'name': 'get_current_time',
            'description': '해당 타임존의 날짜와 시간을 반환합니다.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'timezone': {
                        'type': 'string',
                        'description': '현재 날짜와 시간을 반환할 타임존을 입력하세요. (예: Asia/Seoul)'
                    }
                },
                'required': ['timezone']
            }
        }
    }
]

if __name__ == '__main__':
    get_current_time('America/New_York')

2026-09-20 04:30:40 America/New_York
